In [ ]:
import os
import sys
import json
from dotenv import load_dotenv
from openai import OpenAI

sys.stdout.reconfigure(encoding='utf-8')
load_dotenv()


def section(title):
    bar = '=' * 60
    print()
    print(bar)
    print('  ' + title)
    print(bar)


API_KEY = os.environ.get('OPENAI_API_KEY')
if not API_KEY:
    raise EnvironmentError('OPENAI_API_KEY is not set in your .env file.')

client = OpenAI(api_key=API_KEY)
MODEL = 'gpt-4o-mini'


# -- 1. Basic Text Generation
section('1 - Basic Text Generation')
prompt = 'Explain what a REST API is in exactly three sentences, aimed at a beginner.'
response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': prompt}]
)
print('Prompt : ' + prompt)
print()
print('OpenAI : ' + response.choices[0].message.content.strip())


# -- 2. System Instruction + Generation Config
section('2 - System Instruction + Generation Config')
response2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a witty science communicator. Always answer with enthusiasm and end with a fun fact.'},
        {'role': 'user',   'content': 'Why is the sky blue?'}
    ],
    temperature=0.9,
    max_tokens=200,
)
print('Response:')
print(response2.choices[0].message.content.strip())


# -- 3. Multi-Turn Chat
section('3 - Multi-Turn Chat')
history = []
turns = [
    'I am learning Python. What is the most important concept to master first?',
    'Can you give me a tiny code example of that concept?',
    'How long does it typically take to feel comfortable with it?',
]
for user_msg in turns:
    history.append({'role': 'user', 'content': user_msg})
    resp = client.chat.completions.create(model=MODEL, messages=history)
    assistant_msg = resp.choices[0].message.content.strip()
    history.append({'role': 'assistant', 'content': assistant_msg})
    print()
    print('[User]   ' + user_msg)
    print('[OpenAI] ' + assistant_msg)
print()
print('--- Chat contained ' + str(len(history)) + ' messages total ---')


# -- 4. Structured JSON Output
section('4 - Structured JSON Output')
json_prompt = (
    'List three programming languages popular in 2025. '
    'Respond ONLY with a valid JSON array, no markdown, no extra text. '
    'Each element must have: name (string), primary_use (string), year_created (int).'
)
raw = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': json_prompt}],
    temperature=0.2,
    max_tokens=300,
).choices[0].message.content.strip()
if raw.startswith('```'):
    raw = raw.split('```')[1]
    if raw.startswith('json'):
        raw = raw[4:]
    raw = raw.strip()
languages = json.loads(raw)
print('Parsed languages:')
for lang in languages:
    print('  - ' + lang['name'] + ' (' + str(lang['year_created']) + ') - ' + lang['primary_use'])


# -- 5. Text Embeddings
section('5 - Text Embeddings')
EMBED_MODEL = 'text-embedding-3-small'
sentences = [
    'Machine learning is a subset of artificial intelligence.',
    'Deep learning uses neural networks with many layers.',
    'Pizza is a popular Italian dish with tomato and cheese.',
]
embed_result = client.embeddings.create(model=EMBED_MODEL, input=sentences)
embeddings = [item.embedding for item in embed_result.data]
print('Embedded ' + str(len(embeddings)) + ' sentences.')
print('Each embedding has ' + str(len(embeddings[0])) + ' dimensions.')
print()
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x ** 2 for x in a) ** 0.5
    norm_b = sum(x ** 2 for x in b) ** 0.5
    return dot / (norm_a * norm_b)
sim_ai  = cosine_similarity(embeddings[0], embeddings[1])
sim_off = cosine_similarity(embeddings[0], embeddings[2])
print('Similarity (AI vs Deep Learning) : ' + str(round(sim_ai, 4)))
print('Similarity (AI vs Pizza)         : ' + str(round(sim_off, 4)))
print()
print('-> Related topics score higher - embeddings capture semantic meaning!')


# -- Done
section('All demos completed successfully')
print('API used   : OpenAI API (openai SDK)')
print('Model used : ' + MODEL)
print('Features   : text generation, system instructions, multi-turn chat,')
print('             structured JSON output, text embeddings')
